# Chapitre 9 — Réduire le nombre de dimensions · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch09_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch09_corrige.ipynb`, sur la même page compagnon.

**Convention.** Comme dans le livre, moyennes, variances et covariances divisent par $n$.

**Sommaire.** Données · Les calculs des leçons 9.1 à 9.3 · Exercices 9.1 à 9.6

**Avant d'ouvrir Jupyter :** les exercices de ce chapitre se font d'abord sur papier, avec une calculatrice. Les cellules ci-dessous servent à **vérifier** vos résultats.

**Repères :** bleu = logements ; losange noir = G ; orange = projections ; magenta en tirets = ce que la projection perd.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repères du livre : logements en bleu, G en losange noir, projections en orange,
# ce que la projection perd en magenta (tirets)
BLUE, ORANGE, MAGENTA, INK = '#2a78d6', '#eb6834', '#c2185b', '#0b0b0b'


def project(X, u):
    # coordonnées des points (lignes de X) sur la direction u, ramenée à la norme 1 (équation 9.1)
    u = np.asarray(u, dtype=float)
    return np.asarray(X, dtype=float) @ (u / np.linalg.norm(u))


def cov_matrix(X):
    # matrice de covariance des colonnes de X, avec une division par n (équation 9.2)
    X = np.asarray(X, dtype=float)
    D = X - X.mean(axis=0)
    return D.T @ D / len(X)

## Données

L'identifiant D2.1 est celui du livre et du fichier CSV. Les petits jeux des exercices sont recopiés des énoncés.

In [ ]:
# Données du chapitre 9, recopiées des fichiers CSV du livre (ne pas modifier)

# D2.1 — les sept logements (A à F : ventes de référence ; G : le logement à estimer)
# Colonnes du CSV → noms du notebook : logement → home, surface_m2 → floor_area (m²),
# distance_km → distance (km, distance au centre), prix_kEUR → price (k€)
homes = pd.DataFrame([
    ('A', 32, 3.0, 118),
    ('B', 45, 1.5, 172),
    ('C', 54, 2.5, 185),
    ('D', 70, 1.0, 245),
    ('E', 85, 2.5, 279),
    ('F', 98, 0.5, 335),
    ('G', 62, 2.5, 209)
], columns=['home', 'floor_area', 'distance', 'price'])

reference = homes[homes['home'] != 'G']          # A à F : moyennes et écarts-types de référence
mu = reference[['floor_area', 'distance']].mean().to_numpy()
sigma = reference[['floor_area', 'distance']].std(ddof=0).to_numpy()   # division par n
Z = (reference[['floor_area', 'distance']].to_numpy() - mu) / sigma   # scores standardisés de A à F
z_G = (homes.loc[homes['home'] == 'G', ['floor_area', 'distance']].to_numpy()[0] - mu) / sigma
names = list(reference['home'])
print(homes.to_string(index=False))

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `home` | identifiant du logement (A, B…) | — | `logement` |
| `floor_area` | surface habitable du logement | m² | `surface_m2` |
| `price` | prix de vente | k€ | `prix_kEUR` |
| `distance` | distance au centre-ville | km | `distance_km` |
| `trainee` | identifiant du stagiaire | — | — |
| `math_score` | note de mathématiques | points sur 20 | — |
| `writing_score` | note de rédaction | points sur 20 | — |

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 9.1 à 9.3.

In [ ]:
# Leçon 9.1 — projeter sur un axe (équation 9.1) : axe de la surface, puis diagonale u = (1 ; −1)/√2
u_area = np.array([1.0, 0.0])
u_diag = np.array([1.0, -1.0]) / np.sqrt(2)
t = project(Z, u_diag)
t_G = project(z_G, u_diag)
print(pd.DataFrame({'home': names, 'z_area': Z[:, 0], 'z_distance': Z[:, 1], 't': t}).round(3).to_string(index=False))
print(f"G : t = {t_G:.3f}")
print(f"variance gardée : axe de la surface {np.mean(project(Z, u_area) ** 2):.3f}, diagonale {np.mean(t ** 2):.3f}")
print(f"variance totale {np.sum(Z.var(axis=0)):.3f} = gardée {np.mean(t ** 2):.3f} + perdue {np.sum(Z.var(axis=0)) - np.mean(t ** 2):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, u, title in zip(axes, (u_area, u_diag), ('axe de la surface', 'diagonale')):
    shadows = np.outer(project(Z, u), u)
    for p, s in zip(Z, shadows):
        ax.plot([p[0], s[0]], [p[1], s[1]], color=MAGENTA, linestyle='--', linewidth=1)
    ax.axline((0, 0), u, color=ORANGE, linewidth=1)
    ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, zorder=3)
    ax.scatter(shadows[:, 0], shadows[:, 1], color=ORANGE, zorder=3)
    ax.scatter([z_G[0]], [z_G[1]], marker='D', color=INK, zorder=3)
    for n, p in zip(names + ['G'], list(Z) + [z_G]):
        ax.annotate(n, p, xytext=(4, 4), textcoords='offset points')
    ax.set_aspect('equal'); ax.set_title(title); ax.set_xlabel('z surface'); ax.set_ylabel('z distance')
plt.tight_layout(); plt.show()

In [ ]:
# Leçon 9.2 — la matrice de covariance (équation 9.2) et ses directions propres
products = Z[:, 0] * Z[:, 1]
print('produits z_s × z_d :', products.round(3), '→ somme', round(products.sum(), 3))
C = cov_matrix(Z)
print('matrice de covariance C =\n', C.round(3))
for vec in ([1, 0], [1, -1], [1, 1]):
    print(f"C × {vec} = {(C @ np.array(vec, dtype=float)).round(3)}")
u1, u2 = u_diag
print(f"variance sur la diagonale, avec C seule : {C[0, 0] * u1 ** 2 + 2 * C[0, 1] * u1 * u2 + C[1, 1] * u2 ** 2:.3f}")

In [ ]:
# Leçon 9.3 — l'ACP : valeurs propres rangées par ordre décroissant, part de variance gardée (équation 9.3)
eigenvalues, eigenvectors = np.linalg.eigh(C)          # eigh : matrice symétrique, ordre croissant
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
v1 = eigenvectors[:, 0] * np.sign(eigenvectors[0, 0])  # une direction et son opposée décrivent le même axe
print('valeurs propres :', eigenvalues.round(3), '→ somme', round(eigenvalues.sum(), 3))
print('première direction propre :', v1.round(3))
print('parts gardées (%) :', (100 * eigenvalues / eigenvalues.sum()).round(1))
pc1 = project(Z, v1)
print('ventes rangées par première composante :', [names[i] for i in np.argsort(pc1)],
      '; prix (k€) :', list(reference['price'].to_numpy()[np.argsort(pc1)]))

# B et E : proches sur la composante, éloignés dans le plan
q = project(Z, [1, 1])
iB, iE = names.index('B'), names.index('E')
print(f"B et E : écart sur la composante {abs(pc1[iE] - pc1[iB]):.2f}, distance dans le plan "
      f"{np.linalg.norm(Z[iE] - Z[iB]):.2f}, écart sur la seconde composante {abs(q[iE] - q[iB]):.2f}")

# Pour aller plus loin : la SVD de Z donne les mêmes valeurs propres, s_k² / n
s = np.linalg.svd(Z, compute_uv=False)
print(f"valeurs singulières : {s.round(3)} ; s² / n = {(s ** 2 / len(Z)).round(3)}")

## Exercice 9.1 — Projeter sur deux axes

*Notions : projection et variance gardée (leçon 9.1, équation 9.1).* Trois points, $P_1 = (4 ; 2)$, $P_2 = (1 ; 3)$ et $P_3 = (-2 ; 1)$, et deux directions, $\mathbf{u} = (0{,}8 ; 0{,}6)$ et $\mathbf{v} = (-0{,}6 ; 0{,}8)$.

### a) à d) Sur papier

*Vos calculs :* …

In [ ]:
# Les trois points (une ligne par point) et les deux directions
points = np.array([[4, 2], [1, 3], [-2, 1]], dtype=float)
u = np.array([0.8, 0.6])
v = np.array([-0.6, 0.8])

In [ ]:
# À COMPLÉTER : coordonnées sur u et sur v, puis leurs variances (division par 3)
coords_u = None   # À COMPLÉTER : points @ u
coords_v = None   # À COMPLÉTER
var_u = None      # À COMPLÉTER : np.var(coords_u) divise par n
var_v = None      # À COMPLÉTER

In [ ]:
# Vérification de a) à d)
if coords_u is None or coords_v is None or var_u is None or var_v is None:
    print('pas encore complété. Complétez les quatre lignes.')
else:
    print(f"a) normes {np.linalg.norm(u):.3f} et {np.linalg.norm(v):.3f}, produit scalaire {u @ v:.3f}")
    print('b) sur u :', np.round(coords_u, 2), '; sur v :', np.round(coords_v, 2), '→',
          'correct' if np.allclose(coords_u, [4.4, 2.6, -1.0]) and np.allclose(coords_v, [-0.8, 1.8, 2.0]) else 'à revoir')
    print(f"c) P1 : {coords_u[0] ** 2 + coords_v[0] ** 2:.2f} contre une norme au carré de {points[0] @ points[0]:.2f}")
    total = np.sum(points.var(axis=0))
    print(f"d) variances {var_u:.2f} et {var_v:.2f}, somme {var_u + var_v:.2f}, variance totale {total:.2f} →",
          'correct' if abs(var_u - 5.04) < 1e-9 and abs(var_v - 1.6267) < 1e-3 and abs(var_u + var_v - total) < 1e-9 else 'à revoir')

## Exercice 9.2 — Une matrice de covariance à la main

*Notions : covariance (leçon 1.5), matrice de covariance (leçon 9.2, équation 9.2).* Quatre stagiaires ont passé deux épreuves, notées sur 20 (jeu de données inventé).

- `trainee` — stagiaire, S1 à S4 ;
- `math_score` — note à l'épreuve de calcul, sur 20, en points ;
- `writing_score` — note à l'épreuve de rédaction, sur 20, en points.

In [ ]:
# Les quatre stagiaires (exercices 9.2 à 9.5)
trainees = pd.DataFrame({
    'trainee':       ['S1', 'S2', 'S3', 'S4'],
    'math_score':    [15, 13, 9, 11],
    'writing_score': [11, 13, 9, 7],
})
scores = trainees[['math_score', 'writing_score']].to_numpy(dtype=float)
print(trainees.to_string(index=False))

### a) à c) Sur papier

*Vos calculs :* …

In [ ]:
# À COMPLÉTER : écarts à la moyenne, matrice de covariance (division par 4) et corrélation r
deviations = None   # À COMPLÉTER : scores - scores.mean(axis=0)
M = None            # À COMPLÉTER : deviations.T @ deviations / 4
r = None            # À COMPLÉTER : M[0, 1] / racine de M[0, 0] × M[1, 1]

In [ ]:
# Vérification de a) à c)
if deviations is None or M is None or r is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    print('a) moyennes :', scores.mean(axis=0), '; écarts :', deviations.tolist())
    print('b) et c) M =', np.asarray(M).tolist(), f"; r = {r:.2f} →",
          'correct' if np.allclose(M, [[5, 3], [3, 5]]) and abs(r - 0.6) < 1e-9 else 'à revoir')
    print('contrôle avec cov_matrix :', cov_matrix(scores).tolist())

## Exercice 9.3 — Trouver les directions propres

*Notions : direction propre et valeur propre (leçon 9.2).* On reprend la matrice de l'exercice 9.2, $M = \begin{pmatrix} 5 & 3 \\ 3 & 5 \end{pmatrix}$.

### a) à d) Sur papier

*Vos calculs :* …

In [ ]:
# À COMPLÉTER : valeurs propres de M, rangées par ordre décroissant, et part gardée par la meilleure direction
M93 = np.array([[5.0, 3.0], [3.0, 5.0]])
eigenvalues_93 = None   # À COMPLÉTER : np.linalg.eigh(M93) renvoie (valeurs, vecteurs) en ordre croissant
best_share = None       # À COMPLÉTER : plus grande valeur propre / somme des valeurs propres

In [ ]:
# Vérification de a) à d)
if eigenvalues_93 is None or best_share is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    for vec in ([1, 0], [1, 1], [1, -1]):
        print(f"a) et b) M × {vec} = {(M93 @ np.array(vec, dtype=float)).tolist()}")
    print(f"valeurs propres {eigenvalues_93.round(3).tolist()}, part de la meilleure direction {100 * best_share:.0f} % →",
          'correct' if np.allclose(eigenvalues_93, [8, 2]) and abs(best_share - 0.8) < 1e-9 else 'à revoir')
    r_houses = -0.603
    print('d) r = −0,603 : valeurs propres', np.sort(np.linalg.eigh(np.array([[1, r_houses], [r_houses, 1]]))[0])[::-1].round(3).tolist())

## Exercice 9.4 — Calculer une première composante

*Notions : ACP (leçon 9.3), projection (équation 9.1).* On reprend les écarts à la moyenne de l'exercice 9.2. Ici, on ne standardise pas : les deux épreuves ont la même unité et la même variance. La première direction propre est $\mathbf{v}_1 = (1 ; 1)/\sqrt{2}$.

### a) à d) Sur papier

*Vos calculs et votre interprétation :* …

In [ ]:
# À COMPLÉTER : coordonnées sur v1, variance, puis reconstruction des notes de S1
v1_trainees = np.array([1.0, 1.0]) / np.sqrt(2)
t_trainees = None        # À COMPLÉTER : écarts à la moyenne (exercice 9.2) @ v1_trainees
reconstructed_S1 = None  # À COMPLÉTER : moyennes + t_trainees[0] × v1_trainees

In [ ]:
# Vérification de a) à c)
if t_trainees is None or reconstructed_S1 is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    var_t = np.mean(t_trainees ** 2)
    error = scores[0] - reconstructed_S1
    print('a) coordonnées :', t_trainees.round(2).tolist(), f"; variance {var_t:.2f}")
    print(f"b) part gardée {100 * var_t / np.sum(scores.var(axis=0)):.0f} %")
    print('c) notes reconstruites de S1 :', reconstructed_S1.round(2).tolist(), '; erreur :', error.round(2).tolist(),
          f"; longueur {np.linalg.norm(error):.2f} →",
          'correct' if np.allclose(t_trainees, [2 * np.sqrt(2)] * 2 + [-2 * np.sqrt(2)] * 2)
          and np.allclose(reconstructed_S1, [14, 12]) else 'à revoir')

## Exercice 9.5 — Proches après projection ?

*Notions : limites de la projection (leçon 9.3, équation 9.3).* On reprend les coordonnées de l'exercice 9.4.

### a) à d) Sur papier

*Vos calculs et votre démonstration :* …

In [ ]:
# À COMPLÉTER : distances dans le plan des deux notes
distance_S1_S2 = None   # À COMPLÉTER : np.linalg.norm(scores[0] - scores[1])
distance_S1_S3 = None   # À COMPLÉTER

In [ ]:
# Vérification de a), c) et d)
if distance_S1_S2 is None or distance_S1_S3 is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    t9 = (scores - scores.mean(axis=0)) @ (np.array([1.0, 1.0]) / np.sqrt(2))
    print(f"a) S1 et S2 : écart projeté {abs(t9[0] - t9[1]):.2f}, distance {distance_S1_S2:.2f}")
    print(f"c) S1 et S3 : écart projeté {abs(t9[0] - t9[2]):.2f} ≤ distance {distance_S1_S3:.2f} →",
          'correct' if abs(distance_S1_S2 - np.sqrt(8)) < 1e-9 and abs(distance_S1_S3 - np.sqrt(40)) < 1e-9 else 'à revoir')
    iB, iE = names.index('B'), names.index('E')
    t_BE = project(Z[[iB, iE]], [1, -1])
    q_BE = project(Z[[iB, iE]], [1, 1])
    print(f"d) B et E : écart sur la composante {abs(t_BE[1] - t_BE[0]):.2f}, distance {np.linalg.norm(Z[iE] - Z[iB]):.2f}, "
          f"partie perdue {abs(q_BE[1] - q_BE[0]):.2f}")

## Exercice 9.6 — Défi : l'ACP sans standardiser

*Notions : matrice de covariance (équation 9.2), ACP et unités (leçon 9.3).* Pour les ventes A à F, on travaille sur les écarts à la moyenne bruts, sans standardiser.

- `floor_area` — surface du logement, en m² (jeu D2.1, colonne `surface_m2`) ;
- `distance` — distance au centre, en km (jeu D2.1, colonne `distance_km`).

### a) à d) Sur papier

*Vos calculs et votre conclusion :* …

In [ ]:
# À COMPLÉTER : matrice de covariance brute de A à F (surface en m², distance en km), puis M × w
raw = reference[['floor_area', 'distance']].to_numpy(dtype=float)
M_raw = None   # À COMPLÉTER : cov_matrix(raw)
w = np.array([1.0, -0.0238])
Mw = None      # À COMPLÉTER : M_raw @ w

In [ ]:
# Vérification de a) à d)
if M_raw is None or Mw is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    share = np.linalg.eigvalsh(M_raw)[-1] / np.trace(M_raw)
    raw_m = raw * [1, 1000]                       # distance en mètres
    share_m = np.linalg.eigvalsh(cov_matrix(raw_m))[-1] / np.trace(cov_matrix(raw_m))
    print('a) et b) M =', M_raw.round(3).tolist(), '; M × w =', Mw.round(2).tolist(), f"; rapport {Mw[1] / Mw[0]:.4f}")
    print(f"c) part gardée {100 * share:.1f} % (variance totale {np.trace(M_raw):.2f}) →",
          'correct' if abs(M_raw[0, 1] + 74 / 6) < 1e-9 and abs(Mw[1] / Mw[0] + 0.0238) < 5e-5 else 'à revoir')
    print(f"d) distance en mètres : variance {cov_matrix(raw_m)[1, 1]:.0f} m², part gardée {100 * share_m:.2f} %")